# 📗 DNN 분류 모델 만들기: 손글씨 숫자 분류

교안 01의 직선 모델은 뉴런 하나였습니다. 이 교안은 뉴런 여러 개를 묶은 층을 여러 겹 쌓은 신경망(DNN, deep neural network)으로 손글씨 숫자 그림을 0~9 가운데 하나로 분류합니다. 학습 루프에는 미니배치, 옵티마이저, 학습률 스케줄러를 차례로 붙입니다.

<img src="images/30_lesson02_goals.png" width="1050" alt="제목은 '이번 교안에서 배울 것: 손글씨 숫자를 맞히는 DNN 만들기'입니다. 2행 2열의 판 네 개가 화살표로 이어집니다. 1 DNN 만들기(은닉층과 ReLU) 판에는 784, 128, 64, 10개의 동그라미 층이 선으로 이어진 신경망이 있고, 층마다 세로 점으로 동그라미가 더 있음을 나타냅니다. 2 손실과 정확도(교차 엔트로피, 첫 손실 2.30) 판에는 숫자 0~9 막대 열 개가 모두 같은 높이이고 정답 7 칸만 주황 테두리로 강조돼 있습니다. 3 역전파와 미니배치(층마다 .grad, 64장씩 갱신) 판에는 출력에서 입력 쪽으로 가는 보라 화살표와 카드 묶음이 있습니다. 4 옵티마이저와 스케줄러(SGD, 모멘텀, Adam, warmup + cosine) 판에는 짧게 올라갔다가 곡선으로 내려가는 학습률 그래프가 있습니다.">


**학습 목표** (그림의 번호와 같습니다)

1. **활성화 함수(ReLU·Sigmoid·Tanh·GELU)의 출력 범위와 쓰임을 비교하고, 은닉층과 ReLU가 있는 DNN을 `nn.Sequential`로 만듭니다.**
2. **softmax와 교차 엔트로피 손실, 정확도로 분류 모델을 평가하고, 문제 종류에 맞는 출력층과 손실을 고릅니다.**
3. **역전파가 층마다 구한 기울기를 `.grad`로 확인하고, `DataLoader`로 미니배치 학습 루프를 만듭니다.**
4. **SGD·모멘텀·Adam과 warmup·cosine 스케줄러를 학습 손실로 비교하고, 학습 설정 이름과 연결합니다.**

## 1. 실습 준비

그림을 신경망에 넣으려면 숫자 한 줄로 바꿔야 하므로, 데이터를 불러와 모양부터 확인합니다.

### 1-1. 라이브러리와 데이터

준비 셀은 교안 01과 같습니다(계산 장치와 그래프 글꼴). 데이터는 손글씨 숫자 [MNIST](https://yann.lecun.com/exdb/mnist/)(LeCun, Cortes & Burges, CC BY-SA 3.0)에서 고른 학습용 10,000장과 테스트용 2,000장입니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# torch.load는 텐서 4개(학습용 그림과 정답, 테스트용 그림과 정답)가 든 딕셔너리를 돌려줍니다
mnist = torch.load(data_dir / "본실습" / "mnist_small.pt")
for name in mnist:
    print(name, mnist[name].shape, mnist[name].dtype)

In [ ]:
# 학습용 앞 10장을 정답 숫자와 함께 그립니다
plt.figure(figsize=(10, 1.8))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(mnist["train_images"][i], cmap="gray_r")  # gray_r: 0은 흰색, 255는 검은색
    plt.title(f"정답 {mnist['train_labels'][i].item()}")
    plt.axis("off")
plt.show()

In [ ]:
# 숫자(0~9)마다 학습용 그림이 몇 장인지 셉니다. bincount: 0부터 9까지 각 값이 나온 횟수
label_counts = torch.bincount(mnist["train_labels"])
display(label_counts)

plt.figure(figsize=(6, 2.5))
plt.bar(range(10), label_counts)
plt.xticks(range(10))
plt.xlabel("숫자")
plt.ylabel("그림 수")
plt.title("숫자별 학습용 그림 수")
plt.show()

**결과 읽기**
- **그림:** `uint8`(0~255 정수) 28 × 28 텐서입니다. 0은 바탕, 255는 가장 진한 획입니다.
- **정답:** `int64` 정수 하나(0~9)입니다.
- **숫자별 그림 수:** 878장(숫자 5)에서 1,114장(숫자 7)까지 막대 높이가 비슷합니다. 아무렇게나 하나를 찍으면 약 10%를 맞힙니다.

### 1-2. 픽셀 나누기와 펼치기

`nn.Linear`는 그림 한 장을 숫자 한 줄로 받으므로, 28 × 28 그림을 숫자 784개 한 줄로 바꿉니다.

**펼치기(`reshape`):** 28행을 위에서부터 차례로 옆으로 이어 붙여 숫자 784개(28 × 28)의 한 줄로 만듭니다. 값은 그대로이고 놓인 모양만 바뀝니다.


<img src="images/36_flatten_pixels.png" width="1050" alt="제목은 '그림 한 장을 숫자 784개 한 줄로 펼치기'입니다. 왼쪽에 손글씨 7이 그려진 28 x 28 격자가 있고, 1행, 2행, 28행 띠가 각각 색으로 구분돼 있습니다. 가운데 화살표 위에 'reshape(-1, 784)'가 적혀 있습니다. 오른쪽에는 긴 띠 하나에 '1행: 0~27번', '2행: 28~55번', '...', '28행: 756~783번' 칸이 차례로 이어집니다. 아래 상자에 '픽셀 ÷ 255: 0 → 0.0, 128 → 약 0.5, 255 → 1.0'이 적혀 있습니다.">


1. **펼치기:** 첫 행의 픽셀 28개가 0~27번, 둘째 행이 28~55번, 마지막 28행이 756~783번 자리에 갑니다.
2. **255로 나누기:** 픽셀을 0~1 실수로 바꿉니다. 0은 0.0, 128은 약 0.5, 255는 1.0이 됩니다.
3. **나누는 이유:** 교안 01에서 입력을 표준화한 것과 목적(입력 크기를 줄이기)이 같습니다. 표준화와 달리 평균 0, 표준편차 1이 되지는 않고, 값이 0~1 범위로 줄어듭니다. 입력이 크면 같은 학습률에서도 갱신이 너무 커집니다.

In [ ]:
# 그림 한 장(28 x 28)을 784개 숫자 한 줄로 펼칩니다. -1은 "그림 수는 그대로"라는 뜻입니다
X_train = mnist["train_images"].reshape(-1, 784) / 255  # 255로 나누면 float32 실수가 됩니다
X_test = mnist["test_images"].reshape(-1, 784) / 255
y_train = mnist["train_labels"]
y_test = mnist["test_labels"]

# 데이터는 CPU에 두고, 계산할 때 필요한 만큼만 계산 장치로 옮깁니다
print("X_train:", X_train.shape, X_train.dtype, "| y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, "| y_test:", y_test.shape)
print("픽셀 최솟값:", X_train.min().item(), "| 최댓값:", X_train.max().item())

## 2. 은닉층이 있는 신경망(DNN)

교안 01의 뉴런 하나는 직선 하나만 표현하므로, 픽셀 784개로 숫자 10개를 가르려면 뉴런을 묶은 층을 쌓아야 합니다.

### 2-1. 뉴런 하나에서 층으로

**뉴런 하나:** 입력마다 가중치를 곱해 모두 더하고, 마지막에 절편을 더해 숫자 하나를 냅니다. 교안 01의 `w × x + b`를 입력 여러 개로 늘린 것입니다.


<img src="images/37_neuron_many_inputs.png" width="1050" alt="제목은 '뉴런 하나: 입력 × 가중치를 모두 더하고 절편을 더합니다'입니다. 왼쪽 입력 동그라미 세 개에 '픽셀 0.0', '픽셀 0.5', '픽셀 1.0'이 적혀 있고, 각 화살표 위에 '× 0.2', '× (-0.4)', '× 0.3'이 붙어 가운데 뉴런 원으로 모입니다. 뉴런 원 안에 '더하기 + 절편 0.1'이 있고, 오른쪽 출력에 '0.2'가 있습니다. 아래 계산 띠에 '0.0 × 0.2 + 0.5 × (-0.4) + 1.0 × 0.3 + 0.1 = 0.2'가 적혀 있습니다. 맨 아래 작은 글씨로 'nn.Linear(784, 10): 이런 뉴런 10개가 픽셀 784개를 받습니다'가 있습니다.">


픽셀 3개짜리 작은 예로 계산하면 다음과 같습니다.

1. 픽셀 값 0.0, 0.5, 1.0에 가중치 0.2, -0.4, 0.3을 곱하면 0, -0.2, 0.3입니다.
2. 셋을 더하면 0.1, 절편 0.1을 더하면 뉴런의 출력은 0.2입니다.
3. 실제 손글씨 그림이면 픽셀 784개에 가중치 784개를 곱해 더합니다.

**층(layer):** 같은 입력을 받는 뉴런 여러 개의 묶음입니다. `nn.Linear(784, 10)`은 이런 뉴런 10개라서 숫자 10개를 내고, 가중치는 10 × 784개, 절편은 10개입니다.

In [ ]:
import torch.nn as nn

# 픽셀 784개를 받는 뉴런 10개의 층
torch.manual_seed(0)
layer = nn.Linear(784, 10)
print("weight 모양:", layer.weight.shape, "| bias 모양:", layer.bias.shape)

# 그림 3장을 넣으면 그림마다 숫자 10개가 나옵니다
layer_output = layer(X_train[:3])
print("출력 모양:", layer_output.shape)

### 2-2. 은닉층과 활성화 함수

층을 쌓을 때 층 사이에 활성화 함수를 넣어야 쌓은 효과가 생기므로, 대표 활성화 함수 네 가지를 먼저 비교합니다.

- **은닉층(hidden layer):** 입력과 출력 사이의 층입니다. 은닉층의 뉴런을 은닉 유닛이라고 부릅니다.
- **활성화 함수(activation function):** 층이 계산한 값(곱하고 더한 값)에 한 번 더 씌우는 함수입니다. 직선이 아닌 관계를 만들게 합니다.
- **ReLU:** `max(0, 값)`입니다. 음수는 0, 양수는 그대로입니다. 은닉층의 기본입니다. 양수 쪽 기울기가 늘 1이라 층이 깊어도 기울기가 0으로 줄어드는 문제가 적어 학습이 잘 됩니다.
- **Sigmoid:** 어떤 값이든 0~1 사이로 바꿉니다. 0은 0.5가 됩니다. 0~1이라 이진 분류(예: 악성일 확률) 출력에 씁니다.
- **Tanh:** 어떤 값이든 -1~1 사이로 바꿉니다. 0은 0이 됩니다. 0을 가운데로 둔 S자 모양입니다.
- **GELU:** ReLU를 부드럽게 만든 모양입니다. 0 근처 음수에서는 0 대신 작은 음수를 냅니다. BERT, GPT-2 같은 트랜스포머 모델의 은닉층에서 많이 씁니다.

같은 입력 -2, 0, 2를 넣으면 다음과 같습니다(아래 셀의 표와 같습니다).

1. **ReLU:** 0, 0, 2입니다. 음수만 0이 됩니다.
2. **Sigmoid:** 0.1192, 0.5, 0.8808입니다. 모두 0과 1 사이입니다.
3. **Tanh:** -0.9640, 0, 0.9640입니다. 모두 -1과 1 사이입니다.
4. **GELU:** -0.0455, 0, 1.9545입니다. ReLU(0, 0, 2)와 비슷하지만 조금 부드럽습니다.

In [ ]:
# 같은 입력 -2, 0, 2를 네 활성화 함수에 넣어 출력을 비교합니다
inputs = torch.tensor([-2.0, 0.0, 2.0])
gelu = nn.GELU()  # GELU는 nn.GELU 층으로 씁니다
activation_table = pd.DataFrame({
    "입력": inputs.tolist(),
    "ReLU": torch.relu(inputs).tolist(),
    "Sigmoid": torch.sigmoid(inputs).tolist(),
    "Tanh": torch.tanh(inputs).tolist(),
    "GELU": gelu(inputs).tolist(),
})
display(activation_table.round(4))

In [ ]:
# -4에서 4까지 입력에 대한 네 활성화 함수의 모양
x_values = torch.linspace(-4, 4, 81)
activations = {"ReLU": torch.relu(x_values), "Sigmoid": torch.sigmoid(x_values),
               "Tanh": torch.tanh(x_values), "GELU": gelu(x_values)}

plt.figure(figsize=(18, 4.5))
for k, name in enumerate(activations):
    plt.subplot(1, 4, k + 1)
    plt.plot(x_values, activations[name], linewidth=2)
    plt.axhline(0, color="gray", linewidth=0.8)
    plt.axvline(0, color="gray", linewidth=0.8)
    plt.title(name, fontsize=14)
    plt.xlabel("입력", fontsize=12)
    plt.ylabel("출력", fontsize=12)
plt.tight_layout()
plt.show()

**그래프 읽기**
- **ReLU:** 0 왼쪽은 0으로 평평하고, 오른쪽은 입력 그대로 올라갑니다. 출력에 위쪽 한계가 없습니다.
- **Sigmoid:** 0~1 사이의 S자입니다. 입력 4는 0.98, -4는 0.02처럼 양 끝에서 거의 평평해집니다. 평평한 곳은 기울기가 0에 가까워, 깊은 은닉층에 쓰면 학습이 느려집니다.
- **Tanh:** -1~1 사이의 S자입니다. 양 끝이 평평한 것은 Sigmoid와 같습니다.
- **GELU:** 오른쪽은 ReLU처럼 올라가고, 0 근처 음수에서 살짝 내려갔다가(가장 낮은 값 약 -0.17) 0으로 돌아옵니다.

**ReLU가 필요한 이유:** 직선 계산을 두 번 겹치면 `w2 × (w1 × x + b1) + b2 = (w2 × w1) × x + (w2 × b1 + b2)`로 다시 직선 하나입니다. 말로 하면 "곱하고 더하기"를 두 번 해도 "곱하고 더하기" 한 번과 같다는 뜻입니다. 예: `2x + 1`을 다시 `3 × (2x + 1) + 0`으로 계산하면 `6x + 3`, 여전히 직선입니다. 층 사이에 ReLU를 넣으면 유닛마다 꺾이는 지점이 생겨 직선이 아닌 관계도 표현합니다.


<img src="images/38_relu.png" width="1050" alt="제목은 'ReLU: 음수는 0으로, 양수는 그대로'입니다. 왼쪽에 가로축 '입력', 세로축 '출력' 그래프가 있고, 왼쪽 절반은 0에 붙은 평평한 선, 오른쪽 절반은 0에서 시작해 대각선으로 올라가는 선입니다. 꺾이는 점에 '0'이 표시돼 있습니다. 오른쪽에 예시 상자 두 줄 '-2 → ReLU → 0', '3 → ReLU → 3'이 있습니다. 아래 띠에 '층 사이에 ReLU가 있어야 꺾인 선을 만들 수 있습니다'가 적혀 있습니다.">


가중치를 손으로 정한 작은 모델(입력 1개 -> 은닉 유닛 3개 -> 출력 1개)로 확인합니다. 세 유닛은 `입력 + 2`, `입력`, `입력 - 1.5`를 계산하고, 출력층은 세 값을 1배, -2배, 2배 해서 더합니다.

1. **ReLU 없음:** (x + 2) - 2x + 2(x - 1.5) = x - 1로 직선 하나입니다. 입력 -1이면 -2, 입력 1이면 0입니다.
2. **ReLU 있음, 입력 -1:** 세 유닛 값 1, -1, -2.5가 ReLU를 지나 1, 0, 0이 되고, 출력은 1입니다.
3. **ReLU 있음, 입력 1:** 세 유닛 값 3, 1, -0.5가 ReLU를 지나 3, 1, 0이 되고, 출력은 3 - 2 = 1입니다.
4. 유닛이 0이 되는 입력(-2, 0, 1.5)마다 더해지는 유닛이 바뀌므로, 출력은 그 지점에서 꺾입니다.

In [ ]:
# 입력 1개 -> 은닉 유닛 3개 -> 출력 1개. 꺾이는 지점이 잘 보이게 가중치와 절편을 손으로 정합니다
layer1 = nn.Linear(1, 3)
layer2 = nn.Linear(3, 1)
with torch.no_grad():
    # 유닛 값 = 입력 + 절편(bias)이므로, 입력이 -2, 0, 1.5일 때 각 유닛이 0이 됩니다
    layer1.weight[:] = torch.tensor([[1.0], [1.0], [1.0]])
    layer1.bias[:] = torch.tensor([2.0, 0.0, -1.5])
    # 출력 = 유닛1 x 1 + 유닛2 x (-2) + 유닛3 x 2
    layer2.weight[:] = torch.tensor([[1.0, -2.0, 2.0]])
    layer2.bias[:] = torch.tensor([0.0])

# 같은 두 층을 쓰고, 층 사이에 ReLU가 있는지만 다른 두 모델
no_relu = nn.Sequential(layer1, layer2)
with_relu = nn.Sequential(layer1, nn.ReLU(), layer2)

In [ ]:
# -3에서 3까지 입력 61개를 넣어 출력을 구합니다. 그래프만 그리므로 기울기 기록은 끕니다
x_line = torch.linspace(-3, 3, 61).reshape(-1, 1)
relu = nn.ReLU()
with torch.no_grad():
    no_relu_output = no_relu(x_line)
    with_relu_output = with_relu(x_line)
    hidden_output = relu(layer1(x_line))  # 은닉 유닛 3개가 ReLU를 지난 값, 모양 (61, 3)

# 왼쪽: ReLU 없는 모델, 가운데: 은닉 유닛 3개의 ReLU 출력, 오른쪽: ReLU 있는 모델
plt.figure(figsize=(15, 4.5))
plt.subplot(1, 3, 1)
plt.plot(x_line, no_relu_output, linewidth=2)
plt.title("층 사이에 ReLU 없음", fontsize=13)
plt.xlabel("입력", fontsize=11)
plt.ylabel("출력", fontsize=11)
plt.subplot(1, 3, 2)
for k in range(3):
    plt.plot(x_line, hidden_output[:, k], linewidth=2, label=f"유닛 {k + 1}")
plt.title("은닉 유닛 3개의 ReLU 출력", fontsize=13)
plt.xlabel("입력", fontsize=11)
plt.legend(fontsize=11)
plt.subplot(1, 3, 3)
plt.plot(x_line, with_relu_output, linewidth=2, color="red")
for point in [-2, 0, 1.5]:
    plt.axvline(point, color="gray", linestyle="--")  # 유닛이 0이 되는 지점
plt.title("층 사이에 ReLU 있음", fontsize=13)
plt.xlabel("입력", fontsize=11)
plt.ylabel("출력", fontsize=11)
plt.tight_layout()
plt.show()

**그래프 읽기**
- **ReLU 없음:** 출력은 x - 1인 곧은 직선입니다. 은닉 유닛이 3개여도 직선 하나와 같습니다.
- **은닉 유닛 3개:** 각 유닛은 0이 되는 지점(-2, 0, 1.5) 왼쪽에서는 0이고, 오른쪽에서는 직선으로 올라갑니다.
- **ReLU 있음:** 점선(-2, 0, 1.5)에서 세 번 꺾인 선입니다. 입력 -2까지는 0, 0까지 오르고, 1.5까지 내려갔다가 다시 오릅니다. 학습은 이 꺾이는 위치와 기운 정도를 데이터에 맞게 정합니다.

**활성화 함수 정리:** 이 밖에도 여러 활성화 함수가 있으며, 자주 만나는 네 가지는 다음과 같습니다. 이 교안의 은닉층은 ReLU를 씁니다.

| 이름 | 출력 범위 | 주로 쓰는 곳 | PyTorch |
|---|---|---|---|
| ReLU | 0 이상 | 은닉층의 기본 | `nn.ReLU`, `torch.relu` |
| Sigmoid | 0~1 | 이진 분류 출력을 확률로 읽을 때 | `nn.Sigmoid`, `torch.sigmoid` |
| Tanh | -1~1 | 출력을 -1~1로 맞춰야 할 때 | `nn.Tanh`, `torch.tanh` |
| GELU | 약 -0.17 이상 | BERT, GPT-2 같은 트랜스포머 모델의 은닉층 | `nn.GELU` |

### 2-3. nn.Sequential로 DNN 만들기

층과 ReLU를 차례로 이어 붙이면 DNN이 되므로, 손글씨 숫자를 분류할 모델을 `nn.Sequential`로 만듭니다.

**`nn.Sequential`:** 적은 순서대로 층을 이어 붙여, 입력을 첫 층부터 차례로 통과시키는 모델입니다.


<img src="images/31_dnn_structure.png" width="1050" alt="제목은 'DNN: 픽셀 784개에서 숫자 10개의 점수까지'입니다. 왼쪽에 손글씨 7 그림이 28 x 28 격자로 펼쳐져 '입력 784개'로 들어갑니다. 가운데 '은닉층 1 (128 유닛) + ReLU', '은닉층 2 (64 유닛) + ReLU' 동그라미 기둥 두 개를 지나 오른쪽 '출력층 (10개)'의 0~9 칸에 막대 점수가 붙어 있고, 7 칸의 막대가 가장 깁니다. 맨 아래 띠에 'nn.Linear(784, 128) → ReLU → nn.Linear(128, 64) → ReLU → nn.Linear(64, 10)'이 적혀 있습니다.">


숫자 7 그림 한 장이 이 모델을 지나는 과정입니다.

1. 픽셀 784개가 첫 은닉층(128 유닛)과 ReLU를 지나 숫자 128개가 됩니다.
2. 둘째 은닉층(64 유닛)과 ReLU를 지나 숫자 64개가 됩니다.
3. 출력층이 0~9 각각의 점수 10개를 냅니다. 학습이 잘 되면 7번 칸의 점수가 가장 큽니다.

- **은닉 유닛 수(128, 64):** 정해진 답이 없는 설정값입니다. 784개에서 10개로 줄여 가는 흔한 크기이고, CPU에서도 10에폭 학습이 금방 끝납니다.
- 뒤에서 같은 모양의 새 모델을 여러 번 만들어 비교하므로, 모델 만들기를 함수 `make_model`로 둡니다.

In [ ]:
def make_model():
    """은닉층 두 개(128, 64 유닛)인 DNN을 만들어 계산 장치에 올려 돌려줍니다."""
    return nn.Sequential(
        nn.Linear(784, 128), nn.ReLU(),
        nn.Linear(128, 64), nn.ReLU(),
        nn.Linear(64, 10),
    ).to(device)


# 파라미터 초깃값은 무작위입니다. seed를 고정해 다시 실행해도 같은 모델이 나오게 합니다
torch.manual_seed(0)
model = make_model()
print(model)

In [ ]:
# 층마다 파라미터 모양과 개수. 이름의 앞 숫자는 nn.Sequential 안의 순서(1, 3번 ReLU는 파라미터 없음)
param_rows = []
for name, param in model.named_parameters():
    param_rows.append({"이름": name, "모양": tuple(param.shape), "개수": param.numel()})
param_table = pd.DataFrame(param_rows)
display(param_table)
print("파라미터 수 합계:", param_table["개수"].sum())

**결과 읽기**
- **`0.weight`(128, 784):** 첫 은닉층 유닛 128개가 픽셀 784개마다 가중치 하나씩, 100,352개입니다. 전체의 대부분이 이 층에 있습니다.
- **합계 109,386개:** 교안 01의 직선 모델은 파라미터가 2개였습니다. 개수가 늘어도 학습 방법(손실, 기울기, 갱신)은 같습니다.

### 2-4. 학습 전 순전파와 점수

모델이 무엇을 출력하는지 알아야 손실을 고를 수 있으므로, 학습 전 모델에 그림 3장을 넣어(순전파: 입력에서 출력 쪽으로 계산) 봅니다.

**점수(로짓, logit):** 출력층이 칸(0~9)마다 내는 숫자입니다. 클수록 모델이 그 숫자라고 보는 정도가 큽니다. 확률이 아니므로 음수도 나오고 10개의 합도 1이 아닙니다. 예: 7번 칸 점수가 4.5이고 나머지가 0이면 모델은 7이라고 봅니다.

**한 번에 여러 장:** 모델에는 그림 여러 장을 한꺼번에 넣습니다. 행 하나가 그림 한 장이고, 층을 지날 때마다 열 수(숫자 개수)만 바뀝니다.


<img src="images/50_batch_shapes.png" width="1050" alt="제목은 '그림 여러 장이 층을 지날 때 바뀌는 모양'입니다. 왼쪽부터 세로로 긴 표 네 개가 화살표로 이어집니다. 첫 표 '(64, 784)'에는 '그림 64장 × 픽셀 784개', 둘째 표 '(64, 128)'에는 '그림 64장 × 은닉 유닛 128개', 셋째 표 '(64, 64)'에는 '그림 64장 × 은닉 유닛 64개', 넷째 표 '(64, 10)'에는 '그림 64장 × 점수 10개'가 적혀 있습니다. 화살표 위에 각각 'nn.Linear(784, 128) + ReLU', 'nn.Linear(128, 64) + ReLU', 'nn.Linear(64, 10)'이 있습니다. 아래 띠에 '행 수(그림 수)는 그대로, 열 수만 바뀝니다'가 적혀 있습니다.">


1. 아래 셀은 그림 3장을 넣으므로 입력 모양은 (3, 784)입니다.
2. 은닉층 두 개를 지나며 (3, 128), (3, 64)가 됩니다.
3. 출력은 (3, 10), 그림 3장마다 점수 10개입니다. 미니배치 64장이면 (64, 10)입니다.

In [ ]:
# 그림 3장을 모델에 넣어 그림마다 점수 10개를 받습니다
logits = model(X_train[:3].to(device))
display(logits)
print("정답:", y_train[:3].tolist())

**결과 읽기**
- **모양 (3, 10):** 그림 3장 × 점수 10개입니다.
- **점수 크기:** 학습 전이라 모든 칸이 -0.12~0.18 사이의 비슷한 값입니다. 아직 어떤 숫자도 구별하지 못합니다.
- **`grad_fn`:** 이 값을 만든 계산이 기록돼 있다는 표시입니다. 교안 01처럼 이 기록을 거꾸로 따라가 기울기를 구합니다.

## 3. 분류의 손실과 정확도

교안 01의 MSE는 예측과 정답의 차이를 제곱했지만, 정답 7을 1로 예측한 차이 6에는 뜻이 없습니다. 분류는 정답 칸에 얼마나 높은 확률을 줬는지로 손실을 계산합니다.

### 3-1. 교차 엔트로피

**교차 엔트로피(cross entropy) 손실:** 모델이 정답 칸에 준 확률을 p라 할 때 `-log(p)`입니다. p가 1에 가까우면 손실이 0에 가깝고, p가 0에 가까우면 손실이 크게 커집니다.

- **softmax:** 점수 여러 개를 확률(0~1이고 합이 1인 값)로 바꾸는 함수입니다. 칸마다 e^점수를 구해 그 합으로 나누므로, 점수가 큰 칸일수록 확률이 커집니다.
- `nn.CrossEntropyLoss`는 점수 10개에 softmax를 적용한 뒤 정답 칸의 확률로 -log(p)를 계산합니다. softmax가 손실 안에 들어 있으므로 모델의 출력층은 점수를 그대로 냅니다.
- 정답은 칸 번호(정수)로 줍니다. 예: 정답이 7이면 `7`입니다.
- 여기서 `log`는 자연로그(ln)입니다. 예: log(10) ≈ 2.30, log(2) ≈ 0.69.


<img src="images/35_cross_entropy_digits.png" width="1050" alt="제목은 '교차 엔트로피: 정답 칸의 확률로 매기는 손실'입니다. 왼쪽에 손글씨 7 그림이 있고 '정답: 7'이 붙어 있습니다. 가운데 '학습 전' 판은 0~9 막대가 모두 0.10 높이로 같고 7 칸이 초록 테두리로 강조되며 아래에 '손실 = -log(0.10) = 2.30'이 있습니다. 오른쪽 '학습 후' 판은 7 칸 막대만 0.909로 높고 나머지 아홉 칸은 0.01 높이이며 아래에 '손실 = -log(0.909) = 0.095'가 있습니다. 맨 아래 띠에 '손실 = -log(정답 칸의 확률)'이 적혀 있습니다.">


숫자 7 그림 한 장의 손실이 정해지는 과정입니다(아래 셀의 숫자와 같습니다).

1. **학습 전:** 점수 10개가 모두 0이면 확률은 칸마다 0.10이고, 손실은 -log(0.10) = 2.30입니다.
2. **학습 후:** 7번 칸 점수만 4.5로 커지면 softmax에서 7번 칸은 e^4.5 ≈ 90.0, 나머지 아홉 칸은 e^0 = 1이라 7번 칸 확률은 90.0 ÷ (90.0 + 9) = 0.909입니다. 손실은 -log(0.909) = 0.095입니다.
3. **틀렸을 때:** 3번 칸 점수가 4.5로 커지면 7번 칸 확률은 0.01, 손실은 4.60으로 커집니다.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
answer = torch.tensor([7])  # 정답 칸 번호

# 1) 학습 전: 점수 10개가 모두 0
before_scores = torch.zeros(1, 10)

# 2) 학습 후: 7번 칸 점수만 4.5
after_scores = torch.zeros(1, 10)
after_scores[0, 7] = 4.5

# 3) 틀렸을 때: 3번 칸 점수가 4.5
wrong_scores = torch.zeros(1, 10)
wrong_scores[0, 3] = 4.5

# 학습 후 점수 10개를 softmax로 확률 10개로 바꿉니다(dim=1: 그림 한 장의 10칸 안에서)
after_probs = torch.softmax(after_scores, dim=1)
display(after_probs)
print("확률 10개의 합:", round(after_probs.sum().item(), 4))

In [ ]:
# 세 경우마다 softmax로 구한 정답 칸 확률의 -log와 nn.CrossEntropyLoss 값을 비교합니다
loss_rows = []
for case, scores in [("학습 전", before_scores), ("학습 후", after_scores), ("틀렸을 때", wrong_scores)]:
    answer_prob = torch.softmax(scores, dim=1)[0, 7]  # 정답(7번) 칸의 확률
    loss_rows.append({"경우": case, "정답 칸 확률": answer_prob.item(),
                      "직접 계산 -log(확률)": -torch.log(answer_prob).item(),
                      "nn.CrossEntropyLoss": loss_fn(scores, answer).item()})
loss_table = pd.DataFrame(loss_rows)
display(loss_table.round(4))

In [ ]:
# 정답 칸 확률 p(0.01~1)에 따른 손실 -log(p) 곡선과, 위 표의 세 경우(빨간 점)
p_values = torch.linspace(0.01, 1, 100)
plt.figure(figsize=(5, 3))
plt.plot(p_values, -torch.log(p_values))
plt.scatter(loss_table["정답 칸 확률"], loss_table["nn.CrossEntropyLoss"], color="red")
plt.xlabel("정답 칸 확률 p")
plt.ylabel("손실 -log(p)")
plt.title("정답 칸 확률이 작을수록 손실이 커집니다")
plt.show()

**결과 읽기**
- **softmax 결과:** 7번 칸이 0.9091, 나머지 아홉 칸이 0.0101씩이고 합은 1입니다.
- **두 손실 열:** 세 경우 모두 직접 계산한 -log(확률)과 `nn.CrossEntropyLoss` 값이 같습니다. CrossEntropyLoss는 softmax 뒤 -log(정답 칸 확률)입니다.
- **학습 전:** 정답 칸 확률 0.10, 손실 2.3026입니다. 열 칸에 똑같이 나눠 준 확률입니다.
- **학습 후:** 정답 칸 확률 0.9091, 손실 0.0953입니다.
- **틀렸을 때:** 정답 칸 확률이 0.0101뿐이라 손실이 4.5953으로 학습 전보다도 큽니다. 틀린 칸에 높은 점수를 주면 손실이 가장 큽니다.
- **곡선:** p가 1에서 0.1로 줄 때 손실은 0에서 2.30까지 천천히 오르지만, 0.1 아래에서는 가파르게 커집니다. 빨간 점 셋이 위 표의 세 경우입니다.

### 3-2. 첫 손실 점검

학습을 오래 돌린 뒤에 설정 실수를 발견하면 시간이 낭비되므로, 학습 전 첫 손실이 예상값과 맞는지부터 봅니다.

- 학습 전 모델은 점수가 모두 0 근처라 열 칸에 거의 같은 확률(약 0.10)을 줍니다.
- 그러면 첫 손실은 위의 "학습 전"처럼 log(10) ≈ 2.30 근처여야 합니다.
- 크게 벗어나면 학습 전에 입력 크기(255로 나눴는지), 출력 칸 수, 정답 번호를 먼저 확인합니다.

In [ ]:
import math

# 학습용 10,000장 전체의 첫 손실. 점검만 하므로 기울기 기록은 끕니다
with torch.no_grad():
    first_loss = loss_fn(model(X_train.to(device)), y_train.to(device)).item()
print("첫 손실:", round(first_loss, 4), "| log(10):", round(math.log(10), 4))

# 비교: 255로 나누지 않은 픽셀(0~255)을 그대로 넣으면 첫 손실이 어떻게 되는지 봅니다
raw_pixels = X_train * 255
with torch.no_grad():
    raw_pixel_loss = loss_fn(model(raw_pixels.to(device)), y_train.to(device)).item()
print("0~255 픽셀 그대로일 때 첫 손실:", round(raw_pixel_loss, 4))

**결과 읽기**
- **첫 손실 2.3044:** log(10) = 2.3026과 거의 같습니다. 출력 칸 수와 손실 연결이 맞습니다.
- **0~255 그대로:** 첫 손실이 12.1725로 크게 벗어납니다. 입력이 255배 커서 점수 차이가 벌어지고, 학습 전인데도 틀린 칸에 높은 확률을 준 그림이 많습니다.

### 3-3. argmax 예측과 정확도

손실은 학습에 쓰는 값이고, 사람에게 보고할 때는 몇 장을 맞혔는지가 더 읽기 쉬우므로 정확도도 계산합니다.

- **argmax:** 점수가 가장 큰 칸의 번호입니다. 분류 모델의 예측입니다. 예: 점수가 `[0.1, 2.3, -0.5]`이면 argmax는 1입니다.
- **정확도(accuracy):** 예측이 정답과 같은 그림의 비율입니다. 예: 2,000장 가운데 1,900장을 맞히면 0.95입니다.
- **정확도로 학습하지 않는 이유:** 파라미터를 조금 바꿔도 가장 큰 칸이 바뀌기 전까지 정확도는 그대로라, 기울기로 방향을 알 수 없습니다. 그래서 학습은 손실로 하고, 정확도는 결과를 읽을 때 씁니다.


<img src="images/39_argmax_accuracy.png" width="1050" alt="제목은 '점수에서 예측으로, 예측에서 정확도로'입니다. 왼쪽 판 'argmax'에는 그림 한 장의 점수 막대 10개(0~9)가 있고 7번 막대가 가장 높아 '예측: 7'로 화살표가 이어집니다. 오른쪽 판 '정확도'에는 그림 5장의 표가 있습니다. 열은 '정답', '예측', '맞음'이고 행은 '7, 7, O', '2, 2, O', '1, 1, O', '9, 4, X', '0, 0, O'입니다. 표 아래에 '정확도 = 맞힌 수 ÷ 전체 수 = 4 ÷ 5 = 0.8'이 적혀 있습니다.">


1. **예측:** 그림마다 점수 10개 가운데 가장 큰 칸 번호를 고릅니다. 학습 전 모델은 아래 셀처럼 대부분 0을 고릅니다.
2. **비교:** 예측과 정답이 같으면 맞음, 다르면 틀림입니다.
3. **정확도:** 맞힌 그림 수를 전체 그림 수로 나눕니다. 5장 가운데 4장이면 0.8입니다.

In [ ]:
# 점수가 가장 큰 칸 번호가 예측입니다. dim=1: 그림(행)마다 10칸 가운데 고릅니다
with torch.no_grad():
    predictions = model(X_train.to(device)).argmax(dim=1).cpu()
first_ten = predictions[:10]  # 10,000개 가운데 앞 10개만 봅니다
display(first_ten)

# 정확도 = 맞힌 그림 수 / 전체 그림 수
correct = (predictions == y_train).sum().item()
first_accuracy = correct / len(y_train)
print("맞힌 그림:", correct, "/", len(y_train), "| 학습 전 정확도:", round(first_accuracy, 4))

**결과 읽기**
- **예측 10개:** 10개 가운데 9개가 0입니다. 학습 전 모델은 입력과 관계없이 거의 같은 칸을 고릅니다.
- **정확도 0.1:** 10,000장 가운데 약 1,000장을 맞혔습니다. 아무렇게나 찍을 때의 약 10%와 같은 수준입니다.

### 3-4. 출력층과 손실의 짝

손글씨는 정답이 10가지인 문제였지만, 문제 종류가 바뀌면 출력층 크기와 손실도 함께 바꿔야 하므로 짝을 표로 정리합니다.

- **회귀:** 숫자 하나를 맞히는 문제입니다(교안 01의 시험 점수 예측).
- **이진 분류:** 정답이 두 가지인 문제입니다(악성 또는 양성).
- **다중 분류:** 정답이 셋 이상인 문제입니다(숫자 0~9).


<img src="images/54_output_loss_pairs.png" width="1050" alt="제목은 '문제 종류에 따라 정하는 출력층과 손실'입니다. 가로로 세 칸이 있습니다. 첫 칸 '회귀(시험 점수 예측)'에는 출력 동그라미 1개, '확률로 바꾸지 않음', 'nn.MSELoss / nn.L1Loss', '정답: 실수 (예: 72.5)'가 위에서 아래로 적혀 있습니다. 둘째 칸 '이진 분류(악성/양성)'에는 출력 동그라미 1개, 'Sigmoid → 확률 0~1', 'nn.BCEWithLogitsLoss', '정답: 실수 0.0 또는 1.0'이 있고, 맨 아래 점선 칩에 '또는 출력 2개 + nn.CrossEntropyLoss'가 있습니다. 셋째 칸 '다중 분류(숫자 0~9)'에는 출력 동그라미 10개, 'softmax → 합이 1인 확률 10개', 'nn.CrossEntropyLoss', '정답: 정수 칸 번호 (예: 7)'가 있습니다. 아래 띠에는 'Sigmoid와 softmax는 손실 안에 들어 있으므로 모델 끝에 넣지 않습니다'가 적혀 있습니다.">


| 문제 | 출력층 | 확률로 바꾸는 함수 | 손실 | 정답 |
|---|---|---|---|---|
| 회귀 | 1개 | 없음 | `nn.MSELoss`, `nn.L1Loss` | 실수, 예측과 같은 모양 |
| 이진 분류 | 1개 | Sigmoid | `nn.BCEWithLogitsLoss` | 실수 0.0 또는 1.0 |
| 이진 분류(다른 방법) | 2개 | softmax | `nn.CrossEntropyLoss` | 정답 칸 번호(정수, `torch.long`) |
| 다중 분류 | K개(손글씨는 10개) | softmax | `nn.CrossEntropyLoss` | 정답 칸 번호(정수, `torch.long`) |

- **손실 안의 함수:** `nn.BCEWithLogitsLoss`는 Sigmoid를, `nn.CrossEntropyLoss`는 softmax를 안에서 적용합니다. 그래서 모델의 마지막 층은 점수를 그대로 냅니다.
- **흔한 실수:** 모델 끝에 Sigmoid나 softmax를 넣고 이 손실을 또 쓰면 두 번 적용되어 학습이 느려지거나 멈춥니다. 확률이 필요하면 예측할 때 `torch.sigmoid`, `torch.softmax`를 따로 씁니다.

이진 분류 손실을 점수 하나로 확인합니다. 점수가 2.0인 환자입니다.

1. **확률:** Sigmoid로 악성일 확률은 0.8808입니다.
2. **정답이 악성(1.0)이면:** 정답 확률은 0.8808이고 손실은 -log(0.8808) = 0.1269입니다.
3. **정답이 양성(0.0)이면:** 정답 확률은 1 - 0.8808 = 0.1192이고 손실은 -log(0.1192) = 2.1269입니다.

In [ ]:
# 이진 분류: 출력 점수 하나(점수 2.0)와 nn.BCEWithLogitsLoss
bce_loss_fn = nn.BCEWithLogitsLoss()
score = torch.tensor([2.0])
malignant_prob = torch.sigmoid(score)  # 악성일 확률
print("악성일 확률:", round(malignant_prob.item(), 4))

# 정답이 악성(1.0)일 때와 양성(0.0)일 때, 직접 계산한 -log(정답 확률)과 손실 값을 비교합니다
bce_rows = []
for target_value, answer_prob in [(1.0, malignant_prob), (0.0, 1 - malignant_prob)]:
    target = torch.tensor([target_value])  # 정답은 실수 0.0 또는 1.0
    bce_rows.append({"정답": target_value, "정답 확률": answer_prob.item(),
                     "직접 계산 -log(확률)": -torch.log(answer_prob).item(),
                     "nn.BCEWithLogitsLoss": bce_loss_fn(score, target).item()})
bce_table = pd.DataFrame(bce_rows)
display(bce_table.round(4))

**결과 읽기**
- **두 손실 열:** 정답이 악성이면 0.1269, 양성이면 2.1269로 직접 계산과 `nn.BCEWithLogitsLoss`가 같습니다. 이진 분류 손실도 -log(정답 확률)입니다.
- 함께 따라하기의 유방암 진단은 출력 2개 + `nn.CrossEntropyLoss`(표의 셋째 줄)로 만듭니다.

## 4. 역전파 한 번 보기

파라미터가 10만 개가 넘어도 `loss.backward()` 한 번이면 모든 기울기가 구해지는지, 미니배치 하나로 확인합니다.

### 4-1. 층마다 기울기 확인

**역전파가 층을 지나는 순서:** 교안 01의 체인룰(`u = 2x + 1`, `L = u²`에서 dL/dx = dL/du × du/dx)은 층이 여러 겹이어도 같습니다. 손실에서 가까운 출력층부터 기울기를 구하고, 그 결과를 앞 층으로 넘겨 곱해 가며 입력 쪽 층까지 갑니다.


<img src="images/32_backprop_layers.png" width="1050" alt="제목은 '역전파: 손실에서 입력 쪽으로 층마다 기울기를 구합니다'입니다. 위 줄은 파란 화살표로 '입력 784' → '은닉층 1 (128)' → '은닉층 2 (64)' → '출력층 (10)' → '손실'로 이어지고 '순전파: 값을 앞으로'라고 적혀 있습니다. 아래 줄은 보라 화살표로 '손실'에서 '출력층 .grad (10, 64)', '은닉층 2 .grad (64, 128)', '은닉층 1 .grad (128, 784)' 순서로 왼쪽으로 이어지고 '역전파: 기울기를 뒤로, 체인룰의 곱'이라고 적혀 있습니다. 은닉층 아래 작은 상자에 'ReLU가 0으로 바꾼 유닛은 기울기도 0'이 있고, 오른쪽 아래에 'loss.backward() 한 번'이 있습니다.">


그림 64장으로 역전파가 진행되는 과정입니다.

1. **순전파:** 64장이 세 층을 지나 점수 64 × 10개가 되고, 손실 하나로 평균됩니다.
2. **출력층:** 손실이 점수에 따라 바뀌는 정도에서 출력층 가중치(10 × 64)의 기울기를 구합니다.
3. **은닉층 2와 1:** 출력층이 넘겨준 값에 각 층의 기울기를 곱해 차례로 구합니다. ReLU가 0으로 바꾼 유닛 쪽으로는 0이 넘어갑니다.
4. **결과:** 파라미터 6개 묶음 모두에 `.grad`가 생깁니다.

In [ ]:
# 학습용 앞 64장으로 손실을 계산하고 역전파합니다
X_batch = X_train[:64].to(device)
y_batch = y_train[:64].to(device)
loss = loss_fn(model(X_batch), y_batch)
loss.backward()

# 파라미터마다 기울기 모양과 크기(절댓값 평균)를 표로 봅니다
grad_rows = []
for name, param in model.named_parameters():
    grad_size = param.grad.abs().mean().item()
    grad_rows.append({"이름": name, "파라미터 모양": tuple(param.shape),
                      "기울기 모양": tuple(param.grad.shape), "기울기 절댓값 평균": grad_size})
grad_table = pd.DataFrame(grad_rows)
display(grad_table)

**결과 읽기**
- **모양:** 모든 행에서 기울기 모양이 파라미터 모양과 같습니다. 옵티마이저는 파라미터 하나마다 자기 기울기로 갱신합니다.
- **크기:** 기울기 절댓값 평균이 `0.weight` 0.000404에서 `4.bias` 0.044772까지 100배 넘게 차이 납니다. 학습률 하나로 모든 층을 갱신하면 어느 층에는 크고 어느 층에는 작습니다. 뒤에서 비교할 Adam 옵티마이저가 이 차이를 줄입니다.

### 4-2. 기울기가 0인 가중치

첫 층 가중치의 기울기는 체인룰에 따라 (그 픽셀 값) × (뒤에서 넘어온 값)입니다. 그래서 64장 모두에서 0인 픽셀(가장자리)의 가중치는 기울기도 0이어야 합니다. 64장의 픽셀 합과 첫 층 기울기를 나란히 그려 확인합니다.

In [ ]:
# 픽셀마다 64장의 값을 더합니다. 0이면 64장 모두에서 비어 있는 픽셀입니다
pixel_total = X_batch.sum(dim=0).cpu()  # 모양 (784,)
blank_pixels = (pixel_total == 0).sum().item()

# 첫 층 가중치 기울기(128 x 784)의 절댓값을 픽셀마다 더합니다
first_grad = model[0].weight.grad.abs().sum(dim=0).cpu()  # 모양 (784,)
zero_grad_pixels = (first_grad == 0).sum().item()
print("64장 모두 0인 픽셀:", blank_pixels, "| 기울기가 0인 픽셀:", zero_grad_pixels)

In [ ]:
# 두 값을 28 x 28 그림으로 바꿔 나란히 그립니다
plt.figure(figsize=(6, 3))
plt.subplot(1, 2, 1)
plt.imshow(pixel_total.reshape(28, 28), cmap="gray_r")
plt.title("64장의 픽셀 합")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(first_grad.reshape(28, 28), cmap="gray_r")
plt.title("첫 층 기울기 크기")
plt.axis("off")
plt.show()

**결과 읽기**
- **두 수가 299로 같습니다:** 64장 모두에서 비어 있는 픽셀은 가중치를 바꿔도 손실이 그대로라 기울기가 0입니다.
- **두 그림:** 픽셀 합이 0인 가장자리는 기울기도 흰색(0)이고, 획이 지나가는 가운데에만 기울기가 있습니다. 미니배치마다 들어 있는 그림이 다르므로 갱신되는 가중치도 미니배치마다 다릅니다.

## 5. 미니배치 학습 루프

10,000장을 한꺼번에 계산하면 갱신 한 번이 느리고 메모리도 많이 쓰므로, 데이터를 작은 묶음으로 나눠 묶음마다 갱신합니다.

### 5-1. 미니배치·스텝·에폭

- **미니배치(mini-batch):** 한 번 갱신에 쓰는 데이터 묶음입니다. 묶음의 행 수가 **배치 크기**입니다.
- **스텝(step):** 파라미터 갱신 1번입니다. 미니배치 하나를 쓸 때마다 스텝이 1 늘어납니다.
- **에폭(epoch):** 데이터 전체를 한 번 다 쓰는 것입니다. 에폭마다 순서를 다시 섞어, 매번 같은 그림끼리 묶이지 않게 합니다.
- 무작위로 고른 미니배치의 기울기로 갱신하는 경사하강을 SGD(확률적 경사하강, stochastic gradient descent)라고 부릅니다. 교안 01에서 쓴 `torch.optim.SGD`의 이름이 여기서 왔습니다.


<img src="images/33_batch_epoch_step.png" width="1050" alt="제목은 '미니배치 · 스텝 · 에폭'입니다. 개발자가 손글씨 카드 더미를 섞고, 옆에 '학습용 10,000장을 섞은 뒤 64장씩 나누기'가 적혀 있습니다. 데이터 띠는 64장 칸 여러 개와 맨 끝의 짧은 16장 칸으로 나뉘고, 칸마다 초록 화살표가 갱신 원 하나로 이어지며 첫 원 아래에 '스텝: 파라미터 갱신 1번'이 붙어 있습니다. 띠 위 괄호에 '에폭 1번 = 데이터 전체를 한 번 다 씀 = 157스텝'이 적혀 있습니다. 아래 수식 상자에는 '에폭당 스텝 수 = 올림(10,000 ÷ 64) = 157'과 '전체 스텝 수 = 157 × 10에폭 = 1,570'이 적혀 있습니다.">


학습용 10,000장을 배치 크기 64로 학습하는 과정입니다.

1. **섞기:** 10,000장의 순서를 무작위로 섞습니다.
2. **나누기:** 64장씩 자르면 64장 묶음 156개와 남은 16장 묶음 1개, 모두 157개입니다.
3. **갱신:** 묶음마다 순전파·손실·역전파·갱신을 한 번씩 해서 한 에폭에 157스텝입니다.
4. **반복:** 10에폭이면 157 × 10 = 1,570스텝입니다.

In [ ]:
# 마지막 묶음은 장 수가 모자라도 한 스텝이므로 나눗셈을 올림합니다
batch_size = 64
epochs = 10
steps_per_epoch = math.ceil(len(X_train) / batch_size)
print("에폭당 스텝 수:", steps_per_epoch, "| 전체 스텝 수:", steps_per_epoch * epochs)

### 5-2. TensorDataset과 DataLoader

섞고 나누는 일을 직접 짜지 않아도 되도록, PyTorch가 제공하는 두 도구로 미니배치를 꺼냅니다.

- **`TensorDataset(X, y)`:** 같은 행끼리 짝지어, i번째 그림과 i번째 정답을 함께 꺼내게 묶습니다.
- **`DataLoader(dataset, batch_size=64, shuffle=True)`:** for 문으로 돌 때마다 미니배치(그림 64장, 정답 64개)를 하나씩 줍니다. `shuffle=True`면 에폭마다 순서를 새로 섞습니다.
- **테스트용:** 학습에 쓰지 않고 결과만 계산하므로 섞지 않습니다(`shuffle=False`). 2,000장이면 64장 묶음 31개와 16장 묶음 1개입니다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64, shuffle=False)
print("학습 미니배치 수:", len(train_loader), "| 테스트 미니배치 수:", len(test_loader))

# 첫 미니배치 하나만 꺼내 모양을 봅니다
for X_batch, y_batch in train_loader:
    print("X_batch:", X_batch.shape, "| y_batch:", y_batch.shape)
    break

### 5-3. 학습 함수와 평가 함수

한 에폭 학습과 평가를 이후 모든 실험(배치 크기, 옵티마이저, 스케줄러, 함께 따라하기)에서 되풀이하므로 함수 두 개로 만듭니다.

**`train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None)`**
- 미니배치마다 교안 01과 같은 순서(`zero_grad` -> 손실 -> `backward` -> `step`)로 한 번씩 갱신합니다. `optimizer.zero_grad()`는 지난 스텝의 기울기가 더해지지 않게 지웁니다.
- 학습률 스케줄러(뒤에서 만듭니다)를 넘기면 스텝마다 학습률을 바꿉니다. 넘기지 않으면(`None`) 학습률이 그대로입니다.
- 그 에폭에서 계산한 미니배치 손실의 평균(학습 손실)을 돌려줍니다. 예: 157스텝의 손실 합이 47.1이면 0.30입니다.

**`evaluate(model, loader, loss_fn)`**
- 기울기 없이(`torch.no_grad()`) 예측만 해서 (평균 손실, 정확도)를 돌려줍니다.


<img src="images/51_train_one_epoch.png" width="1050" alt="제목은 'train_one_epoch: 미니배치마다 네 단계를 반복합니다'입니다. 큰 테두리 안에 상자 네 개가 시계 방향으로 이어집니다: 1 'optimizer.zero_grad(): 지난 기울기 지우기', 2 'loss = loss_fn(model(X_batch), y_batch): 손실 계산', 3 'loss.backward(): 기울기 구하기', 4 'optimizer.step(): 파라미터 갱신'. 원 가운데에는 '157번 반복 = 1에폭'이 있습니다. 오른쪽의 '미니배치 64장' 카드 묶음에서 나온 화살표가 2번 상자로 들어갑니다. 테두리 아래로 나온 화살표 옆에 '157번 끝나면'이 붙어 있고, 화살표는 '돌려주는 값: 손실 157개의 평균' 상자에 닿습니다.">


1. `train_loader`에서 미니배치(그림 64장, 정답 64개)를 하나 꺼냅니다.
2. 네 단계(기울기 지우기, 손실 계산, 기울기 구하기, 갱신)를 한 번 합니다. 이것이 스텝 1번입니다.
3. 미니배치 157개를 다 쓰면 한 에폭이 끝나고, 손실 157개의 평균을 돌려줍니다.

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

### 5-4. 10에폭 학습하기

함수가 준비됐으므로 새 모델을 10에폭 학습하고, 에폭마다 학습 손실과 학습 데이터 정확도를 기록합니다.

- **옵티마이저:** 뒤에서 비교합니다. 여기서는 PyTorch에서 가장 많이 쓰는 Adam을 기본 학습률 0.001로 씁니다.
- **학습 데이터 정확도:** 에폭이 끝날 때마다 3-3절처럼 학습용 10,000장을 한 번에 모델에 넣어 argmax 예측으로 계산합니다. `train_loader`를 한 번 더 돌지 않으므로, 미니배치를 섞는 순서가 뒤의 같은 설정 실험(Adam 0.001)과 똑같이 유지됩니다.
- 테스트 데이터는 학습을 마친 뒤 한 번만 씁니다.

In [ ]:
# 학습 전 상태에서 시작하도록 새 모델과 옵티마이저를 만듭니다
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
model = make_model()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 에폭마다 학습하고, 학습 데이터 정확도를 기록합니다
history_rows = []
for epoch in range(1, 11):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)

    # 학습용 10,000장을 한 번에 넣어 예측하고 정확도를 구합니다(3-3절과 같은 방법)
    with torch.no_grad():
        train_predictions = model(X_train.to(device)).argmax(dim=1).cpu()
    train_accuracy = (train_predictions == y_train).sum().item() / len(y_train)

    # 에폭 번호, 학습 손실, 학습 데이터 정확도를 한 행으로 기록합니다
    history_rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 데이터 정확도": train_accuracy})
history_table = pd.DataFrame(history_rows)
display(history_table.round(4))

In [ ]:
# 에폭별 학습 손실과 학습 데이터 정확도
plt.figure(figsize=(9, 3))
plt.subplot(1, 2, 1)
plt.plot(history_table["에폭"], history_table["학습 손실"], marker="o")
plt.xticks(range(1, 11))
plt.xlabel("에폭")
plt.ylabel("학습 손실")
plt.title("학습 손실")
plt.subplot(1, 2, 2)
plt.plot(history_table["에폭"], history_table["학습 데이터 정확도"], marker="o", color="green")
plt.xticks(range(1, 11))
plt.xlabel("에폭")
plt.ylabel("정확도")
plt.title("학습 데이터 정확도")
plt.tight_layout()
plt.show()

**결과 읽기**
- **학습 손실:** 1에폭 0.8256에서 10에폭 0.0752로 줄었습니다. 첫 손실 2.30에서 출발해 첫 에폭에 가장 많이 줄고, 뒤로 갈수록 천천히 줄어듭니다.
- **학습 데이터 정확도:** 1에폭 뒤 0.8974에서 10에폭 뒤 0.9879로 올랐습니다. 학습 전 0.1에서 크게 올랐습니다.

### 5-5. 테스트 데이터로 한 번 확인하기

학습 데이터 정확도는 모델이 이미 본 그림의 결과이므로, 처음 보는 테스트 2,000장으로 한 번 확인하고 틀린 그림을 봅니다.

In [ ]:
# 학습을 마친 모델을 테스트 데이터로 한 번 평가합니다
test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
print("테스트 손실:", round(test_loss, 4), "| 테스트 정확도:", round(test_accuracy, 4))

In [ ]:
# 테스트 그림의 예측을 구하고, 틀린 그림의 번호를 모읍니다
with torch.no_grad():
    test_predictions = model(X_test.to(device)).argmax(dim=1).cpu()

wrong_index = []
for i in range(len(y_test)):
    if test_predictions[i] != y_test[i]:
        wrong_index.append(i)
print("틀린 그림:", len(wrong_index), "/", len(y_test))

In [ ]:
# 틀린 그림 앞 10장을 정답과 예측으로 그립니다
plt.figure(figsize=(10, 2.2))
for k in range(10):
    i = wrong_index[k]
    plt.subplot(1, 10, k + 1)
    plt.imshow(mnist["test_images"][i], cmap="gray_r")
    plt.title(f"정답 {y_test[i].item()}\n예측 {test_predictions[i].item()}")
    plt.axis("off")
plt.show()

**결과 읽기**
- **테스트 정확도 0.948:** 학습 데이터 정확도(0.9879)보다 낮습니다. 틀린 비율로 보면 학습 데이터는 약 1.2%, 테스트는 5.2%로 4배가 넘습니다. 두 값의 차이를 진단하는 방법은 다음 시간에 배웁니다.
- **틀린 그림 104장:** 앞 10장 가운데 4장을 5로 잘못 읽었습니다(정답 1, 9, 9, 6). 옆에 점이 찍힌 1, 꼬리가 휘거나 획이 가는 9, 아래 고리가 작은 6처럼 흔한 모양과 조금 다르게 쓴 숫자입니다.
- 틀린 그림의 모양은 모델이 어떤 그림을 틀리는지 이해하는 근거가 됩니다. 비슷한 필체가 자주 틀리면 그런 그림을 더 모을지 정할 때도 씁니다.

### 5-6. 배치 크기의 영향

배치 크기를 바꾸면 같은 에폭에서도 스텝 수가 달라지므로, 배치 크기 세 가지를 같은 3에폭으로 학습해 학습 손실을 비교합니다.


<img src="images/16_batch_size.png" width="1050" alt="제목은 '배치 크기의 영향: 같은 에폭이라도 달라지는 것'입니다. 왼쪽 '작은 배치' 판은 데이터 띠가 작은 칸 여러 개로 나뉘고 '에폭당 갱신이 많음', '같은 에폭에 손실이 더 줄어듦', 'GPU 메모리를 적게 씀'이 적혀 있습니다. 오른쪽 '큰 배치' 판은 데이터 띠가 큰 칸 두 개로 나뉘고 '에폭당 갱신이 적음', '갱신마다 더 많은 그림을 봄', 'GPU 메모리를 많이 씀'이 적혀 있습니다. 두 판 아래의 손실 그래프는 같은 축에서 같은 높이로 시작합니다. 작은 배치 곡선은 빨리 내려가 점선 아래에서 끝나고, 큰 배치 곡선은 천천히 내려가 점선 높이에서 끝납니다. 아래 띠에는 '배치 크기를 바꾸면 학습률을 다시 고릅니다'가 적혀 있습니다.">


배치 크기 16과 256의 차이를 숫자로 보면 다음과 같습니다.

1. **배치 16:** 에폭당 625스텝, 3에폭이면 1,875번 갱신합니다. 갱신마다 그림 16장만 봅니다.
2. **배치 256:** 에폭당 40스텝, 3에폭이면 120번 갱신합니다. 갱신마다 그림 256장을 봅니다.
3. 같은 3에폭이어도 갱신 횟수가 약 16배 차이 납니다.

In [ ]:
import time

batch_losses = {}
batch_rows = []
for size in [16, 64, 256]:
    torch.manual_seed(0)
    size_model = make_model()
    size_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=size, shuffle=True)
    size_optimizer = torch.optim.Adam(size_model.parameters(), lr=0.001)

    # 같은 3에폭을 학습하며 에폭별 학습 손실과 걸린 시간을 기록합니다
    start = time.perf_counter()
    losses = []
    for epoch in range(3):
        losses.append(train_one_epoch(size_model, size_loader, loss_fn, size_optimizer))
    seconds = time.perf_counter() - start

    batch_losses[size] = losses
    batch_rows.append({"배치 크기": size, "에폭당 스텝 수": len(size_loader),
                       "3에폭 학습 손실": losses[-1], "학습 시간(초)": seconds})
batch_table = pd.DataFrame(batch_rows)
display(batch_table.round({"3에폭 학습 손실": 4, "학습 시간(초)": 2}))

In [ ]:
# 배치 크기별 에폭마다의 학습 손실
plt.figure(figsize=(5, 3))
for size in [16, 64, 256]:
    plt.plot([1, 2, 3], batch_losses[size], marker="o", label=f"배치 {size}")
plt.xticks([1, 2, 3])
plt.xlabel("에폭")
plt.ylabel("학습 손실")
plt.title("배치 크기별 학습 손실")
plt.legend()
plt.show()

**결과 읽기**
- **학습 손실:** 3에폭 뒤 배치 16이 0.1777로 가장 낮고, 256이 0.3657로 가장 높습니다. 같은 에폭에서는 갱신을 많이 한 쪽이 손실을 더 줄였습니다.
- **학습 시간:** 스텝이 많은 배치 16이 가장 오래 걸립니다(초는 컴퓨터마다 다릅니다). GPU는 한 미니배치 안의 그림을 동시에 계산하므로, GPU에서는 큰 배치가 에폭당 시간이 더 짧고 대신 메모리를 많이 씁니다.
- **실무 판단:** 배치 크기는 보통 GPU 메모리가 허락하는 범위에서 정하고, 배치 크기를 바꾸면 스텝 수가 바뀌므로 학습률과 에폭 수를 다시 고릅니다.

## 6. 옵티마이저

역전파 결과에서 본 것처럼 층마다 기울기 크기가 달라, 같은 학습률로 기울기만 빼는 규칙은 어느 층에는 너무 크고 어느 층에는 너무 작습니다. 그래서 갱신 규칙(옵티마이저)을 바꿔 비교합니다.

### 6-1. SGD·모멘텀·Adam

**옵티마이저(optimizer):** 기울기로 파라미터를 갱신하는 규칙입니다. `optimizer.step()`이 이 규칙대로 파라미터를 바꿉니다.

- **SGD:** 이번 미니배치 기울기만 써서 `파라미터 - 학습률 × 기울기`로 갱신합니다. 교안 01에서 쓴 규칙입니다.
- **모멘텀(momentum):** 이전 갱신 방향을 누적해 두고(속도) 이번 기울기를 더해 갱신합니다. 미니배치마다 방향이 바뀌는 성분은 서로 지워지고, 같은 방향이 이어지는 성분은 쌓여 갱신이 커집니다.
- **Adam:** 모멘텀처럼 기울기의 평균을 쓰면서, 파라미터마다 기울기 제곱 평균의 제곱근(기울기의 대략적인 크기)으로 나눠 갱신 크기를 맞춥니다. [Kingma & Ba, 2014](https://arxiv.org/abs/1412.6980)가 제안했습니다.


<img src="images/34_optimizers.png" width="1050" alt="제목은 '옵티마이저: 기울기로 파라미터를 갱신하는 규칙'이고 같은 타원 등고선 판 세 개에서 손실이 가장 작은 지점(별)까지 가는 경로를 비교합니다. SGD 판은 '지금 기울기만 보고 갱신'이며 좁은 방향으로 크게 지그재그하며 점이 많습니다. 모멘텀 판은 '이전 갱신 방향의 누적을 더함'이며 점점 빠르게 나아가다 별을 조금 지나쳤다가 돌아옵니다. Adam 판은 '모멘텀 + 파라미터마다 갱신 크기 조절'이며 거의 곧은 선으로 다섯 점 만에 닿습니다. 아래 띠에 'PyTorch: torch.optim.SGD, SGD(momentum=0.9), torch.optim.Adam'이 적혀 있습니다.">


기울기 크기가 다른 두 가중치를 갱신하는 예입니다(역전파 결과 표의 첫 층과 출력층처럼).

1. 첫 층의 가중치 하나는 기울기가 매번 0.001, 출력층의 가중치 하나는 매번 0.1이라고 합시다.
2. **SGD(학습률 0.1):** 두 가중치가 스텝마다 0.0001과 0.01만큼 바뀝니다. 100배 차이입니다.
3. **Adam(학습률 0.001):** 기울기를 기울기 제곱 평균의 제곱근(여기서는 0.001과 0.1)으로 나누므로 두 가중치 모두 스텝마다 약 0.001씩 바뀝니다.

| 옵티마이저 | 갱신에 쓰는 것 | PyTorch 코드 |
|---|---|---|
| SGD | 이번 미니배치 기울기 | `torch.optim.SGD(model.parameters(), lr=0.1)` |
| 모멘텀 | 이전 갱신 방향의 누적 + 이번 기울기 | `torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)` |
| Adam | 기울기의 평균 ÷ 파라미터마다 기울기 제곱 평균의 제곱근(기울기의 대략적인 크기) | `torch.optim.Adam(model.parameters(), lr=0.001)` |

- **라이브러리 기본 학습률:** PyTorch 2.13의 `torch.optim.Adam`과 `torch.optim.SGD`는 `lr`을 적지 않으면 0.001을 씁니다. ([PyTorch Adam 문서](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html), [SGD 문서](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html))
- **`momentum=0.9`:** 이전 속도를 90% 남긴다는 뜻으로 흔히 쓰는 값입니다. 기본값은 0(모멘텀 없음)입니다.

### 6-2. 옵티마이저와 학습률 비교

옵티마이저마다 잘 맞는 학습률이 다르므로, 교안 01의 방법대로 10배 간격 후보 네 개를 모두 시도합니다. 같은 초깃값에서 5에폭씩 학습하고 학습 손실로 비교합니다.

In [ ]:
optimizer_losses = {}
for optimizer_name in ["SGD", "모멘텀", "Adam"]:
    for lr in [0.0001, 0.001, 0.01, 0.1]:
        # 모든 조합이 같은 초깃값과 같은 미니배치 순서로 시작합니다
        torch.manual_seed(0)
        compare_model = make_model()
        if optimizer_name == "SGD":
            compare_optimizer = torch.optim.SGD(compare_model.parameters(), lr=lr)
        elif optimizer_name == "모멘텀":
            compare_optimizer = torch.optim.SGD(compare_model.parameters(), lr=lr, momentum=0.9)
        else:
            compare_optimizer = torch.optim.Adam(compare_model.parameters(), lr=lr)

        # 5에폭 동안 에폭별 학습 손실을 기록합니다
        losses = []
        for epoch in range(5):
            losses.append(train_one_epoch(compare_model, train_loader, loss_fn, compare_optimizer))
        optimizer_losses[(optimizer_name, lr)] = losses

In [ ]:
# 5에폭 뒤 학습 손실을 표로 봅니다(행은 학습률, 열은 옵티마이저)
optimizer_rows = []
for lr in [0.0001, 0.001, 0.01, 0.1]:
    row = {"학습률": lr}
    for optimizer_name in ["SGD", "모멘텀", "Adam"]:
        row[optimizer_name] = optimizer_losses[(optimizer_name, lr)][-1]
    optimizer_rows.append(row)
optimizer_table = pd.DataFrame(optimizer_rows)
display(optimizer_table.round(4))

In [ ]:
# 옵티마이저마다 학습률별 학습 손실 곡선을 그립니다(세 그래프의 세로축 범위를 같게)
plt.figure(figsize=(16, 4.8))
for k, optimizer_name in enumerate(["SGD", "모멘텀", "Adam"]):
    plt.subplot(1, 3, k + 1)
    # 0.0001은 점선으로 다른 선 위에 그립니다(SGD에서는 0.001 선과 거의 겹칩니다)
    for lr, style, layer in zip([0.0001, 0.001, 0.01, 0.1], ["--", "-", "-", "-"], [3, 2, 2, 2]):
        plt.plot(range(1, 6), optimizer_losses[(optimizer_name, lr)], marker="o", linewidth=2,
                 linestyle=style, zorder=layer, label=f"lr={lr}")
    plt.ylim(0, 3.6)  # 위쪽 여백(2.5 위)에 범례를 둡니다
    plt.xticks(range(1, 6), fontsize=11)
    plt.yticks(fontsize=11)
    plt.xlabel("에폭", fontsize=12)
    plt.ylabel("학습 손실", fontsize=12)
    plt.title(optimizer_name, fontsize=14)
    plt.legend(loc="upper right", ncol=2, fontsize=11)
plt.tight_layout()
plt.show()

**결과 읽기**
- **SGD:** 0.1에서만 손실이 잘 줄어 0.2685입니다. 0.001 이하에서는 5에폭 동안 첫 손실 2.30 근처(2.28~2.30)에 머물러, 0.0001(점선)과 0.001 선이 거의 겹칩니다.
- **모멘텀:** 0.1에서 0.1088, 0.01에서 0.2721입니다. `momentum=0.9`는 같은 방향이 이어질 때 갱신을 약 10배(1 ÷ (1 - 0.9))로 키우므로, 같은 학습률에서는 SGD보다 빨라 보입니다. 실제로 모멘텀 0.01(0.2721)은 SGD 0.1(0.2685)과, 모멘텀 0.001(1.4307)은 SGD 0.01(1.3572)과 비슷합니다.
- **Adam:** 0.001에서 0.1752, 0.01에서 0.1025입니다. 0.1에서는 손실이 1.4~1.6에서 더 줄지 않아 수렴하지 않습니다.
- **비교:** 학습률을 옵티마이저마다 맞게 고르면 모멘텀과 Adam이 비슷하게 빠르고 SGD가 가장 느립니다. 같은 옵티마이저 안에서 학습률에 따른 차이가 옵티마이저 사이의 차이보다 큽니다.

**Adam의 학습률 고르기** (교안 01의 규칙: 정한 횟수 안에 수렴한 후보 가운데 발산 경계에서 가장 먼 값)
1. **수렴하지 않음:** 0.1은 손실이 줄지 않았으므로 발산 경계는 0.1 근처입니다.
2. **5에폭 안에 손실이 충분히 줄어든 후보:** 0.001과 0.01입니다(둘 다 아직 조금씩 내려가는 중). 0.0001은 0.4747로 아직 높습니다.
3. **선택:** 두 후보 가운데 경계(0.1)에서 가장 먼 **0.001**입니다. Adam의 기본 학습률과 같습니다.
4. **실측과의 차이:** 5에폭 학습 손실만 보면 0.01(0.1025)이 0.001(0.1752)보다 낮습니다. 그래도 0.01은 수렴하지 않은 0.1과 10배 차이뿐이라, 데이터나 모델이 바뀌면 발산 쪽으로 넘어갈 여유가 적습니다. 그래서 규칙대로 경계에서 먼 값을 고릅니다.
- 학습 손실이 낮은 설정이 학습에 쓰지 않은 데이터에서도 나은지는 다음 시간에 확인합니다.

**실무 순서**
- 옵티마이저는 Adam으로 두고, 학습률은 기본값 0.001 근처의 10배 간격 후보로 먼저 고릅니다.
- 옵티마이저를 바꾸면 학습률도 다시 고릅니다. SGD에 Adam의 학습률 0.001을 그대로 쓰면 5에폭 동안 거의 학습되지 않습니다.

## 7. 학습률 스케줄러

학습률 하나를 끝까지 고정하면 초반의 큰 갱신과 후반의 세밀한 갱신을 함께 맞추기 어려우므로, 학습 도중 스텝에 따라 학습률을 바꿉니다.

### 7-1. warmup과 cosine 감쇠

**학습률 스케줄러(learning rate scheduler):** 스텝마다 학습률을 바꾸는 규칙입니다. LLM 학습은 warmup 구간과 감쇠 구간을 이어 붙인 스케줄을 많이 씁니다.

- **warmup:** 처음 몇 스텝 동안 작은 학습률에서 최고 학습률까지 직선으로 올립니다. 학습 초반에는 파라미터가 무작위라 기울기가 크고 방향도 자주 바뀌어, 갱신이 크면 손실이 갑자기 커질 수 있기 때문입니다. ([Goyal et al., 2017](https://arxiv.org/abs/1706.02677))
- **cosine 감쇠:** warmup 뒤 남은 스텝 동안 코사인 곡선을 따라 0 근처까지 줄입니다. 후반에 갱신이 작아져 손실이 흔들리지 않고 더 낮은 곳으로 내려갑니다. ([Loshchilov & Hutter, 2016](https://arxiv.org/abs/1608.03983))


<img src="images/17_lr_schedule.png" width="1050" alt="제목은 '학습률 스케줄러: warmup으로 올리고 감쇠로 줄이기'입니다. 가로축 스텝, 세로축 학습률 그래프에서 짧은 주황 warmup 구간은 학습률이 0 근처에서 꼭대기까지 직선으로 오르고, 긴 파랑 감쇠 구간은 코사인 모양으로 내려가 0에 닿습니다. 왼쪽 카드는 'warmup: 처음엔 작은 갱신 크기'(초반에는 기울기가 크고 방향이 자주 바뀌어 갱신 크기가 크면 손실이 커질 수 있습니다), 오른쪽 카드는 '감쇠: 후반엔 갱신 크기를 줄임'(학습률이 작아져 마지막 에폭들의 손실이 덜 흔들립니다)입니다. 아래 회색 띠에 'Hugging Face 기본값: lr_scheduler_type = linear, warmup_steps = 0'이 적혀 있습니다.">


10에폭(157 × 10 = 1,570스텝), 최고 학습률 0.01, warmup 157스텝(첫 1에폭)일 때 학습률이 바뀌는 과정입니다.

1. **첫 스텝:** 최고 학습률의 1%인 0.0001에서 시작합니다.
2. **warmup:** 157스텝 동안 직선으로 올라 최고 학습률 0.01에 닿습니다.
3. **감쇠:** 남은 1,413스텝 동안 코사인 곡선을 따라 줄어듭니다. 감쇠 구간의 가운데쯤에서 0.005(절반)가 됩니다.
4. **마지막 스텝:** 0에 아주 가까운 값이 됩니다.

### 7-2. 스케줄러 만들기

PyTorch는 warmup과 cosine을 각각 스케줄러로 제공하므로, 둘을 이어 붙여 하나로 만듭니다.

- **`LinearLR`:** 옵티마이저의 학습률에 곱하는 배율을 `start_factor`에서 `end_factor`까지 `total_iters`스텝 동안 직선으로 바꿉니다(warmup). `start_factor`는 0이면 에러가 나므로 0.01처럼 0보다 큰 값을 씁니다.
- **`CosineAnnealingLR`:** `T_max`스텝 동안 학습률을 코사인 곡선으로 줄입니다(감쇠).
- **`SequentialLR`:** `milestones`에 적은 스텝에서 앞 스케줄러에서 뒤 스케줄러로 넘어갑니다.
- 최고 학습률은 옵티마이저를 만들 때 준 `lr`입니다. 스케줄러는 `train_one_epoch` 안에서 스텝마다 `scheduler.step()`으로 한 칸씩 진행합니다.

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR


def make_warmup_cosine(optimizer, total_steps, warmup_steps):
    """warmup_steps 동안 학습률을 올리고, 남은 스텝 동안 cosine으로 줄이는 스케줄러를 돌려줍니다."""
    # warmup: 최고 학습률의 1%에서 시작해 warmup_steps 동안 100%까지 직선으로 올립니다
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)

    # 감쇠: 남은 스텝 동안 코사인 곡선을 따라 0 근처까지 줄입니다
    cosine = CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)

    # warmup_steps번째 스텝에서 warmup에서 cosine으로 넘어갑니다
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_steps])

In [ ]:
# 전체 스텝 수 = 에폭 수 x 에폭당 스텝 수, warmup은 그 10%(첫 1에폭)
epochs = 10
total_steps = epochs * len(train_loader)
warmup_steps = total_steps // 10
print("전체 스텝:", total_steps, "| warmup 스텝:", warmup_steps)

In [ ]:
# 학습률이 바뀌는 모양만 보려고 만든 모델과 옵티마이저입니다(최고 학습률 0.01)
preview_model = make_model()
preview_optimizer = torch.optim.Adam(preview_model.parameters(), lr=0.01)
preview_scheduler = make_warmup_cosine(preview_optimizer, total_steps, warmup_steps)

# 학습 없이 스텝만 넘기며 스텝마다 쓸 학습률을 기록합니다
lr_history = []
for step in range(total_steps):
    lr_history.append(preview_scheduler.get_last_lr()[0])  # 이번 스텝의 학습률(파라미터 묶음이 하나라 [0])
    preview_optimizer.step()  # 기울기가 없어 파라미터는 바뀌지 않습니다. 스케줄러는 optimizer.step() 뒤에 부릅니다
    preview_scheduler.step()

print("첫 스텝:", lr_history[0], "| warmup 끝:", lr_history[warmup_steps], "| 마지막 스텝:", f"{lr_history[-1]:.8f}")

In [ ]:
# 스텝별 학습률. 점선이 warmup이 끝나는 스텝입니다
plt.figure(figsize=(6, 3))
plt.plot(lr_history)
plt.axvline(warmup_steps, color="gray", linestyle="--")
plt.xlabel("스텝")
plt.ylabel("학습률")
plt.title("warmup + cosine (최고 학습률 0.01)")
plt.show()

### 7-3. 고정 학습률과 비교

스케줄러가 결과를 실제로 낫게 하는지는 최고 학습률을 같게 두고 스케줄만 바꿔 학습해야 알 수 있습니다. 앞에서 고른 0.001과, 학습 손실은 더 낮았지만 발산 경계에 가까워 고르지 않은 0.01 두 가지에서 고정과 warmup + cosine을 10에폭씩 비교합니다.

In [ ]:
schedule_losses = {}
for peak_lr in [0.001, 0.01]:
    for schedule_name in ["고정", "warmup + cosine"]:
        torch.manual_seed(0)
        schedule_model = make_model()
        schedule_optimizer = torch.optim.Adam(schedule_model.parameters(), lr=peak_lr)

        # 고정이면 스케줄러 없이(None) 학습합니다
        scheduler = None
        if schedule_name == "warmup + cosine":
            scheduler = make_warmup_cosine(schedule_optimizer, total_steps, warmup_steps)

        # 10에폭 동안 에폭별 학습 손실을 기록합니다
        losses = []
        for epoch in range(epochs):
            losses.append(train_one_epoch(schedule_model, train_loader, loss_fn, schedule_optimizer, scheduler))
        schedule_losses[(peak_lr, schedule_name)] = losses

In [ ]:
# 네 실험의 10에폭 학습 손실
schedule_rows = []
for peak_lr in [0.001, 0.01]:
    for schedule_name in ["고정", "warmup + cosine"]:
        losses = schedule_losses[(peak_lr, schedule_name)]
        schedule_rows.append({"최고 학습률": peak_lr, "스케줄": schedule_name, "10에폭 학습 손실": losses[-1]})
schedule_table = pd.DataFrame(schedule_rows)
display(schedule_table.round(4))

In [ ]:
# 최고 학습률마다 두 스케줄의 학습 손실 곡선을 겹쳐 그립니다
plt.figure(figsize=(13, 4.8))
for k, peak_lr in enumerate([0.001, 0.01]):
    plt.subplot(1, 2, k + 1)
    for schedule_name in ["고정", "warmup + cosine"]:
        plt.plot(range(1, epochs + 1), schedule_losses[(peak_lr, schedule_name)], marker="o", linewidth=2, label=schedule_name)
    plt.yscale("log")  # 0.003과 0.1처럼 크기가 다른 손실을 함께 보려고 로그 축을 씁니다
    plt.xticks(range(1, epochs + 1), fontsize=11)
    plt.yticks(fontsize=11)
    plt.xlabel("에폭", fontsize=12)
    plt.ylabel("학습 손실(로그 축)", fontsize=12)
    plt.title(f"최고 학습률 {peak_lr}", fontsize=14)
    plt.legend(loc="lower left", fontsize=11)
plt.tight_layout()
plt.show()

**결과 읽기**
- **최고 학습률 0.01:** 고정은 6에폭 0.0774에서 7에폭 0.0876으로 한 번 다시 커졌고, 10에폭에 0.0565입니다. warmup + cosine은 후반에 학습률이 줄어 꾸준히 내려가 0.0027까지 갑니다. 네 실험 가운데 가장 낮습니다.
- **최고 학습률 0.001:** warmup + cosine이 0.1107로 고정(0.0752)보다 오히려 높습니다. 학습률이 이미 작은데 더 줄이니, 아직 줄어들 손실이 남았는데도 갱신이 작아졌습니다.
- **실무 판단:** 스케줄러는 후반 갱신을 줄여 주므로, 고정 학습률에서는 경계에 가까워 고르지 않은 더 큰 최고 학습률(0.01)도 후반에 꾸준히 내려가게 합니다. 스케줄러를 붙이면 최고 학습률을 다시 고릅니다.
- 학습 손실이 더 낮은 이 설정이 학습에 쓰지 않은 데이터에서도 나은지는 다음 시간에 확인합니다.

**확인 퀴즈**: 학습 손실이 후반에 같은 높이에서 오르내리기만 하고 더 줄지 않습니다. 발산하지는 않았습니다. 학습률 설정에서 무엇을 바꿔 볼 수 있을까요?

<details><summary>정답 보기</summary>

후반으로 갈수록 학습률을 줄이는 감쇠 스케줄(cosine 등)을 붙입니다.

- **원인:** 손실은 최솟값 근처까지 줄었지만 학습률이 그대로라, 미니배치마다 갱신이 커서 손실이 오르내립니다.
- **감쇠의 효과:** 학습률이 줄면 갱신도 작아져, 최고 학습률 0.01 실험처럼 손실이 오르내리지 않고 더 낮게 내려갑니다.

</details>

### 7-4. 학습 설정 이름과 오늘 만든 코드

LLM 파인튜닝은 Hugging Face `Trainer`로 많이 하고, 학습 설정은 `TrainingArguments`에 이름으로 적습니다. 오늘 쓴 코드가 어느 이름인지 알면 설정을 읽고 바꿀 수 있습니다.

| 설정 이름 | 오늘 코드에서 | 기본값 |
|---|---|---|
| `learning_rate` | 옵티마이저의 `lr`(스케줄러를 쓰면 최고 학습률) | 5e-5 |
| `lr_scheduler_type` | 감쇠 모양(`CosineAnnealingLR`이면 `"cosine"`) | `"linear"` |
| `warmup_steps` | `LinearLR`의 `total_iters`. 0.1처럼 1보다 작으면 전체 스텝 대비 비율 | 0 |
| `per_device_train_batch_size` | `DataLoader`의 `batch_size` | 8 |
| `num_train_epochs` | 에폭 수 | 3 |

- 기본값은 transformers 5.13 기준입니다. ([TrainingArguments 문서](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments))
- 이미 학습된 LLM을 파인튜닝할 때는 기본값 5e-5처럼 오늘(0.001 안팎)보다 훨씬 작은 학습률을 씁니다.

## 8. 🖐️ 함께 따라하기: 유방암 진단 모델

손글씨 그림이 아닌 표 데이터(환자 한 명 = 측정값 30개)에서도 같은 DNN과 학습 루프가 작동하는지, 직접 작성해 확인합니다.

**본실습과 다른 점**
- **입력:** 세포 사진에서 얻은 측정값 30개(반지름, 질감, 둘레 등)입니다([UCI Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic), CC BY 4.0). 열마다 크기가 제각각이라 255로 나누는 대신 표준화합니다.
- **정답:** 두 가지(0 = 양성, 1 = 악성)입니다. 출력 칸이 2개일 뿐 같은 `nn.CrossEntropyLoss`를 씁니다. 이진 분류라 출력 1개 + `nn.BCEWithLogitsLoss`(정답은 실수 0.0/1.0)로도 만들 수 있습니다.
- **나누기:** 학습용과 테스트용이 나뉘어 있지 않으므로 직접 8 : 2로 나눕니다.
- **판단:** 정확도와 함께, 악성인데 양성으로 예측한 환자 수(놓친 악성)를 셉니다.

**실습 진행 방법**
- 작성 셀은 이 절에만 있습니다. 셀에 남은 `...`를 주석에 맞춰 채웁니다. 나머지 코드는 제공합니다.
- `...`가 남은 채로 실행하면 에러가 납니다. 작성 셀을 완성해 실행하면 같은 셀에서 결과를 볼 수 있습니다.
- 먼저 직접 작성해 봅니다. 완성 코드는 수업이 끝난 뒤 강사가 공유합니다.

**설정**
- 학습용 80%, 테스트용 20%(순서를 섞는 seed 0)
- DNN: 측정값 30개 → 은닉 유닛 16개(ReLU) → 점수 2개
- Adam 학습률 0.001, 배치 크기 32, 20에폭

### 8-1. 데이터 읽기

In [ ]:
# 환자 569명: 측정값 30열과 악성 여부(1 = 악성, 0 = 양성)
cancer = pd.read_csv(data_dir / "따라하기" / "breast_cancer.csv")
print("행 수:", len(cancer), "| 열 수:", cancer.shape[1])
display(cancer.head())  # 569행 가운데 앞 5행

In [ ]:
# 악성과 양성 환자 수
diagnosis_counts = cancer["악성"].value_counts()
display(diagnosis_counts)

### 8-2. 텐서 만들기, 나누기, 표준화

테스트용 환자는 학습 때 보지 못한 새 환자라고 가정하므로, 표준화의 평균과 표준편차도 학습용 데이터에서만 구해 두 데이터에 똑같이 씁니다.

- **텐서 만들기:** 교안 01처럼 `torch.tensor(df[열 목록].to_numpy(), dtype=..., device=device)`로 만듭니다. 입력은 `torch.float32`, 정답 번호는 정수 `torch.long`입니다.
- **정답 모양:** 정답은 본실습 `y_train`처럼 1차원 (569,) 모양이어야 하므로, 열 이름 하나를 대괄호 한 겹으로 고릅니다(`cancer["악성"]`). 두 겹(`cancer[["악성"]]`)이면 (569, 1)이 되어 손실을 계산할 때 에러가 납니다.
- **나누기:** `torch.randperm(n, generator=...)`는 0부터 n-1까지를 무작위로 섞은 번호를 돌려줍니다. 예: n이 5면 `[3, 0, 4, 1, 2]` 같은 순서입니다. 같은 seed면 언제나 같은 순서입니다.
- **번호로 행 고르기:** `X[train_index]`는 `train_index`에 든 번호의 행만 모읍니다. 예: 번호가 `[3, 0]`이면 3번 행과 0번 행입니다.
- **표준화:** 열마다 (값 - 학습용 평균) ÷ 학습용 표준편차입니다. 말로 하면 "평균에서 표준편차 몇 개만큼 떨어져 있나"입니다. 예: 평균 14, 표준편차 3.5인 열에서 17.5는 1.0, 10.5는 -1.0이 됩니다.


<img src="images/52_standardize_train_stats.png" width="1050" alt="제목은 '표준화: 평균과 표준편차는 학습용 데이터에서만 구합니다'입니다. 왼쪽 위 '학습용 환자 80%' 상자에서 화살표가 가운데 '평균 14, 표준편차 3.5' 상자로 갑니다. 가운데 상자에서 화살표 두 개가 오른쪽 위 '학습용 표준화'와 오른쪽 아래 '테스트용 표준화'로 갑니다. 왼쪽 아래 '테스트용 환자 20%' 상자는 오른쪽 아래로만 이어지고, 가운데 상자로 가는 길에는 빨간 X와 '평균 계산에 쓰지 않음'이 있습니다. 아래 띠에 '(값 - 평균) ÷ 표준편차: 17.5 → 1.0, 10.5 → -1.0'이 적혀 있습니다.">


1. 학습용 환자 455명으로 열마다 평균과 표준편차를 구합니다.
2. 그 평균과 표준편차로 학습용 455명을 표준화합니다.
3. 테스트용 114명도 같은 평균과 표준편차로 표준화합니다. 테스트용으로 새로 구하지 않습니다.

In [ ]:
# 측정값 30열의 이름(마지막 열 악성 앞까지)
feature_columns = list(cancer.columns[:30])

# 🖐️ 함께 따라하기: 측정값 30열은 입력 텐서로, 악성 열은 정답 텐서로 만드세요
cancer_features = ...  # float32 실수
cancer_labels = ...    # long 정수, 1차원 (569,)
print("입력:", cancer_features.shape, cancer_features.dtype, "| 정답:", cancer_labels.shape, cancer_labels.dtype)

In [ ]:
# 섞은 번호로 앞 80%는 학습용, 나머지는 테스트용으로 쓸 번호입니다
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(cancer), generator=generator)
n_train = int(len(cancer) * 0.8)
train_index = order[:n_train]
test_index = order[n_train:]

# 🖐️ 함께 따라하기: 번호로 학습용과 테스트용을 나누고, 학습용 평균과 표준편차로 표준화하세요
X_cancer_train = ...
X_cancer_test = ...
y_cancer_train = ...
y_cancer_test = ...

train_mean = ...  # 학습용 열별 평균
train_std = ...   # 학습용 열별 표준편차
X_cancer_train = ...
X_cancer_test = ...

print("학습용:", X_cancer_train.shape, "| 테스트용:", X_cancer_test.shape)
print("표준화한 학습용 첫 열의 평균:", round(X_cancer_train[:, 0].mean().item(), 4))

### 8-3. DataLoader와 DNN 준비

본실습과 같은 순서(DataLoader, 모델, 옵티마이저, 첫 손실 점검)로 준비합니다. 정답이 두 가지이므로 첫 손실은 log(10)이 아니라 log(2) ≈ 0.69 근처여야 합니다.

In [ ]:
# 🖐️ 함께 따라하기: 학습용과 테스트용 DataLoader를 만드세요(학습용만 섞습니다)
cancer_train_loader = ...
cancer_test_loader = ...

# 🖐️ 측정값 30개 -> 은닉 유닛 16개(ReLU) -> 점수 2개인 DNN과 Adam 옵티마이저를 만드세요
torch.manual_seed(0)  # 모델 초깃값과 미니배치 섞는 순서를 고정합니다
cancer_model = ...
cancer_optimizer = ...

# 학습 전 첫 손실을 구해 log(2)와 비교하세요(loss_fn은 본실습의 nn.CrossEntropyLoss)
with torch.no_grad():
    cancer_first_loss = ...
print("첫 손실:", round(cancer_first_loss, 4), "| log(2):", round(math.log(2), 4))

**결과 읽기:** 첫 손실 0.8059는 log(2) = 0.6931보다 조금 큽니다. 학습 전 점수가 0에서 조금 벗어나 있어서 생기는 차이로, 이 정도는 정상입니다. 본실습에서 0~255 픽셀을 그대로 넣은 12.17처럼 크게 벗어나면 입력과 정답을 점검합니다.

### 8-4. 학습하기

본실습의 `train_one_epoch`를 그대로 씁니다. 에폭마다 학습 손실을 기록해 손실이 줄어드는지 봅니다.

In [ ]:
# 🖐️ 함께 따라하기: 20에폭 학습하며 에폭마다 학습 손실을 기록하세요
cancer_rows = []
for epoch in range(1, 21):
    train_loss = ...
    cancer_rows.append({"에폭": epoch, "학습 손실": train_loss})
cancer_history = pd.DataFrame(cancer_rows)
display(cancer_history.round(4))

In [ ]:
# 에폭별 학습 손실 곡선(점선은 첫 손실 기준 log(2))
plt.figure(figsize=(5, 3))
plt.plot(cancer_history["에폭"], cancer_history["학습 손실"], marker="o", label="학습 손실")
plt.axhline(math.log(2), color="gray", linestyle="--", label="log(2) = 0.69")
plt.xticks(range(2, 21, 2))
plt.xlabel("에폭")
plt.ylabel("학습 손실")
plt.title("유방암 진단 모델의 학습 손실")
plt.legend()
plt.show()

**결과 읽기:** 학습 손실이 1에폭 0.7421에서 20에폭 0.0932로 줄었습니다. 16에폭 뒤로는 0.0888~0.1034 사이에서 조금씩 오르내립니다.

### 8-5. 테스트 정확도와 놓친 악성 환자

진단 모델에서 악성을 양성으로 놓치면 치료가 늦어지고, 양성을 악성으로 잘못 알리면 추가 검사를 한 번 더 합니다. 두 실수의 비용이 다르므로, 테스트 데이터로 한 번 평가할 때 정확도와 함께 둘을 따로 셉니다.

**두 가지 실수:** 정답(악성, 양성)과 예측(악성, 양성)을 짝지으면 네 경우가 나오고, 그 가운데 두 경우가 실수입니다.


<img src="images/53_two_mistakes.png" width="1050" alt="제목은 '진단 모델의 두 가지 실수'입니다. 2행 2열 표가 있고 행 머리는 '정답: 악성', '정답: 양성', 열 머리는 '예측: 악성', '예측: 양성'입니다. 칸은 '맞힘 37명'(정답 악성, 예측 악성), '놓친 악성 3명'(정답 악성, 예측 양성, 빨간 칸), '잘못 알린 악성 5명'(정답 양성, 예측 악성, 주황 칸), '맞힘 55명'(정답 양성, 예측 양성)입니다. 표 위에 '예시: 환자 100명'이 있고, 아래 띠에 '정확도 = (37 + 55) ÷ 100 = 0.92, 그래도 놓친 악성 3명은 따로 봅니다'가 적혀 있습니다.">


환자 100명(악성 40명, 양성 60명)을 예로 들면 다음과 같습니다.

1. 악성 40명 가운데 37명을 악성으로 맞히고 3명을 양성으로 놓쳤습니다(놓친 악성 3명).
2. 양성 60명 가운데 55명을 맞히고 5명을 악성으로 잘못 알렸습니다(잘못 알린 악성 5명).
3. 정확도는 (37 + 55) ÷ 100 = 0.92이지만, 이 숫자만으로는 놓친 악성 3명이 보이지 않습니다.

In [ ]:
# 🖐️ 함께 따라하기: 테스트 데이터로 한 번 평가해 정확도를 구하세요
cancer_test_loss, cancer_test_accuracy = ...
print("테스트 정확도:", round(cancer_test_accuracy, 4))

In [ ]:
# 🖐️ 함께 따라하기: 테스트 환자의 예측(argmax)을 구하고 두 가지 실수를 세세요
with torch.no_grad():
    cancer_predictions = ...
missed_malignant = ...  # 정답은 악성(1)인데 양성(0)으로 예측한 환자 수
false_alarm = ...       # 정답은 양성(0)인데 악성(1)으로 예측한 환자 수

# 비교: 모든 환자를 양성이라고 답할 때의 정확도
benign_count = (y_cancer_test == 0).sum().item()
print("테스트 환자:", len(y_cancer_test), "| 그중 악성:", y_cancer_test.sum().item())
print("놓친 악성:", missed_malignant, "| 잘못 알린 악성:", false_alarm)
print("모두 양성이라고 답할 때 정확도:", round(benign_count / len(y_cancer_test), 4))
print("모두 양성이라고 답할 때 놓친 악성:", y_cancer_test.sum().item())

**생각해 보기**: 정확도가 높은 진단 모델이라도 정확도만 보고 쓰면 안 되는 이유는 무엇일까요? 위 결과의 숫자로 설명해 봅니다.

<details><summary>정답 보기</summary>

정확도는 두 실수를 한 숫자로 합쳐, 비용이 큰 실수가 몇 번인지 보여 주지 않기 때문입니다.

- **모두 양성이라고 답하는 모델:** 아무것도 학습하지 않아도 정확도가 약 61%입니다. 그런데 테스트용 악성 환자 44명을 모두 놓칩니다.
- **학습한 DNN:** 정답본 실행에서는 정확도가 98%대이고, 틀린 2명 가운데 1명이 놓친 악성입니다. 같은 98%라도 틀린 2명이 모두 놓친 악성인 모델과 모두 잘못 알린 악성인 모델은 병원에서 쓸 가치가 다릅니다.
- **실무 판단:** 진단처럼 실수의 비용이 다른 문제는 정확도와 함께 놓친 악성 수를 따로 보고합니다. 여러 모델 가운데 하나를 고를 때는 테스트 데이터가 아니라 따로 떼어 둔 데이터로 비교합니다(다음 시간에 배웁니다).

</details>

## 9. 핵심 정리

- **활성화 함수:** ReLU(음수는 0, 은닉층 기본), Sigmoid(0~1, 이진 분류 확률), Tanh(-1~1), GELU(부드러운 ReLU, BERT와 GPT-2 같은 트랜스포머 모델의 은닉층)입니다.
- **DNN:** `nn.Linear` 층 사이에 ReLU를 넣어 쌓은 신경망입니다. ReLU가 없으면 층을 쌓아도 직선 하나와 같습니다. 이 교안의 모델은 784 → 128 → 64 → 10, 파라미터 109,386개였습니다.
- **분류의 손실과 정확도:** `nn.CrossEntropyLoss`는 점수에 softmax를 적용한 뒤 정답 칸에 준 확률 p로 -log(p)를 계산합니다. 학습 전 첫 손실은 log(정답 종류 수) 근처여야 합니다(10가지면 2.30, 2가지면 0.69). 예측은 점수의 `argmax`이고, 학습은 손실로, 보고는 정확도로 합니다.
- **출력층과 손실의 짝:** 회귀는 출력 1개 + `nn.MSELoss`, 이진 분류는 출력 1개 + `nn.BCEWithLogitsLoss`(또는 출력 2개 + `nn.CrossEntropyLoss`), 다중 분류는 출력 K개 + `nn.CrossEntropyLoss`입니다. Sigmoid와 softmax는 손실 안에 있으므로 모델 끝에 넣지 않습니다.
- **역전파:** `loss.backward()` 한 번에 출력층에서 입력 쪽 층으로 체인룰이 이어져, 모든 파라미터의 `.grad`가 파라미터와 같은 모양으로 채워집니다.
- **미니배치 학습 루프:** 에폭당 스텝 수 = 올림(데이터 수 ÷ 배치 크기)입니다. `DataLoader`가 섞고 나누며, `train_one_epoch`는 미니배치마다 `zero_grad` → 손실 → `backward` → `step`을 합니다. 설정 비교는 학습 손실로 하고, 테스트 데이터는 학습을 마친 모델에 한 번만 씁니다.
- **옵티마이저:** 옵티마이저는 Adam(기본 학습률 0.001)으로 두고 학습률부터 고릅니다. 옵티마이저나 배치 크기를 바꾸면 학습률을 다시 고릅니다.
- **학습률 스케줄러:** warmup으로 초반 갱신을 작게, cosine 감쇠로 후반 갱신을 작게 합니다. 스케줄러를 붙이면 최고 학습률을 다시 고릅니다. `TrainingArguments`의 `learning_rate`, `lr_scheduler_type`, `warmup_steps`가 오늘 만든 코드의 이름입니다.

다음 시간에는 그림의 위치 정보를 그대로 쓰는 신경망(CNN)을 만들고, 학습 데이터에만 맞춰지는 과적합을 찾아 줄이는 방법을 배웁니다.